# Home Credit 스파이크 1단계 — application_train.csv 단독

목적: 연구 질문("데이터 복잡도에 따라 선형/비선형 모델 성능 격차가 어떻게 달라지는가")을
GMC보다 한 단계 복잡한 데이터에서 확인한다. 이번 1단계는 **다중 테이블 조인 없이** 주 테이블
(`application_train.csv`, 122컬럼, 범주형 16개, 결측 다수)만으로 먼저 비교한다.
다중 테이블 조인·집계 피처는 2단계 스파이크에서 별도로 다룬다.

GMC 스파이크 결과(기준선): 홀드아웃 AUC 로지스틱 회귀 0.8598 vs 튜닝 XGBoost 0.8691 (+0.0094),
상위 5% 포착 정밀도 47.47% vs 48.00%(+0.5%p). (`docs/spike-feasibility.md`)


In [1]:
import time
from contextlib import contextmanager

import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import randint, uniform
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42
DATA_DIR = "../../data/raw_home_credit"
TARGET = "TARGET"

timings = {}


@contextmanager
def timer(step_name):
    start = time.perf_counter()
    yield
    elapsed = time.perf_counter() - start
    timings[step_name] = elapsed
    print(f"[{step_name}] {elapsed:.1f}초")


## 1. 데이터 로드 및 구조 확인

In [2]:
with timer("데이터 로드"):
    train = pd.read_csv(f"{DATA_DIR}/application_train.csv", index_col="SK_ID_CURR")

print("shape:", train.shape)
print("타깃 양성 비율:", f"{train[TARGET].mean():.2%}")

cat_cols = train.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in train.columns if c not in cat_cols + [TARGET]]
print(f"범주형 변수: {len(cat_cols)}개 / 수치형 변수: {len(num_cols)}개")
print("(GMC는 변수 10개, 전부 수치형이었음)")


[데이터 로드] 0.8초
shape: (307511, 121)
타깃 양성 비율: 8.07%
범주형 변수: 16개 / 수치형 변수: 104개
(GMC는 변수 10개, 전부 수치형이었음)


/var/folders/yk/y8jtmy9n3mqfnj05f__7_wqr0000gn/T/ipykernel_92136/1589528562.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train.select_dtypes(include="object").columns.tolist()


In [3]:
missing = train.isna().mean().sort_values(ascending=False)
print("결측률 상위 10개:")
print(missing.head(10))
print()
print(f"결측률 50% 이상 컬럼 수: {(missing >= 0.5).sum()}개")
print(f"결측이 전혀 없는 컬럼 수: {(missing == 0).sum()}개")


결측률 상위 10개:
COMMONAREA_AVG              0.698723
COMMONAREA_MODE             0.698723
COMMONAREA_MEDI             0.698723
NONLIVINGAPARTMENTS_AVG     0.694330
NONLIVINGAPARTMENTS_MODE    0.694330
NONLIVINGAPARTMENTS_MEDI    0.694330
FONDKAPREMONT_MODE          0.683862
LIVINGAPARTMENTS_MEDI       0.683550
LIVINGAPARTMENTS_AVG        0.683550
LIVINGAPARTMENTS_MODE       0.683550
dtype: float64

결측률 50% 이상 컬럼 수: 41개
결측이 전혀 없는 컬럼 수: 54개


## 2. 알려진 이상값: `DAYS_EMPLOYED`의 365243 코드값

GMC의 연체 변수 96/98 코드값과 같은 패턴 — 특정 값이 "해당 없음"을 나타내는 placeholder로 쓰였다.


In [4]:
print(train["DAYS_EMPLOYED"].describe())
anomaly = (train["DAYS_EMPLOYED"] == 365243).sum()
print(f"\n365243(이상값) 건수: {anomaly:,} ({anomaly / len(train):.1%})")
print("이 값을 가진 행의 타깃 양성 비율:", train.loc[train['DAYS_EMPLOYED'] == 365243, TARGET].mean())
print("전체 타깃 양성 비율:", train[TARGET].mean())


count    307511.000000
mean      63815.045904
std      141275.766519
min      -17912.000000
25%       -2760.000000
50%       -1213.000000
75%        -289.000000
max      365243.000000
Name: DAYS_EMPLOYED, dtype: float64

365243(이상값) 건수: 55,374 (18.0%)
이 값을 가진 행의 타깃 양성 비율: 0.05399646043269404
전체 타깃 양성 비율: 0.08072881945686496


## 3. 최소 전처리

GMC 스파이크와 동일한 원칙(결측 중앙값 대체, 알려진 코드값 처리, 극단값 클리핑)에 범주형 인코딩을 추가한다.


In [5]:
with timer("전처리"):
    df = train.copy()

    # 알려진 코드값 처리: DAYS_EMPLOYED 365243 -> 결측
    df.loc[df["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = np.nan

    # 수치형 결측 중앙값 대체
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())

    # 범주형 결측은 "Missing" 범주로 채운 뒤 원-핫 인코딩
    df[cat_cols] = df[cat_cols].fillna("Missing")
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

    # 수치형 극단값 클리핑 (원-핫 컬럼은 0/1이라 대상에서 자동 제외됨 - 분위수가 0/1 근처라 영향 없음)
    feature_cols = [c for c in df.columns if c != TARGET]
    lower = df[feature_cols].quantile(0.005)
    upper = df[feature_cols].quantile(0.995)
    df[feature_cols] = df[feature_cols].clip(lower=lower, upper=upper, axis=1)

print("전처리 후 피처 수:", len(feature_cols), "(원-핫 인코딩으로 증가)")
print("결측 합계:", df[feature_cols].isna().sum().sum())


[전처리] 1.3초
전처리 후 피처 수: 234 (원-핫 인코딩으로 증가)
결측 합계: 0


## 4. 학습/홀드아웃 분할

In [6]:
X = df[feature_cols]
y = df[TARGET]

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("학습:", X_train.shape, "홀드아웃:", X_holdout.shape)


학습: (246008, 234) 홀드아웃: (61503, 234)


## 5. 로지스틱 회귀 — 하이퍼파라미터 탐색

In [7]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

logreg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
logreg_grid = {
    "clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "clf__class_weight": [None, "balanced"],
}

with timer("로지스틱 회귀 탐색"):
    logreg_search = GridSearchCV(logreg_pipe, logreg_grid, scoring="roc_auc", cv=cv, n_jobs=-1)
    logreg_search.fit(X_train, y_train)

logreg_best = logreg_search.best_estimator_
logreg_proba = logreg_best.predict_proba(X_holdout)[:, 1]
logreg_auc = roc_auc_score(y_holdout, logreg_proba)
print("최적 파라미터:", logreg_search.best_params_)
print("홀드아웃 AUC:", round(logreg_auc, 4))


/Users/gene/creditLens/ml/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[로지스틱 회귀 탐색] 57.3초
최적 파라미터: {'clf__C': 0.01, 'clf__class_weight': 'balanced'}
홀드아웃 AUC: 0.7487


## 6. XGBoost — 하이퍼파라미터 탐색

In [8]:
pos_weight_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight 후보 비율:", round(pos_weight_ratio, 2))

xgb_param_dist = {
    "n_estimators": randint(100, 400),
    "max_depth": randint(2, 8),
    "learning_rate": uniform(0.01, 0.29),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_weight": randint(1, 10),
    "scale_pos_weight": [1, 5, 10, round(pos_weight_ratio, 2)],
}

with timer("XGBoost 탐색"):
    xgb_search = RandomizedSearchCV(
        xgb.XGBClassifier(objective="binary:logistic", eval_metric="auc", random_state=RANDOM_STATE),
        param_distributions=xgb_param_dist,
        n_iter=25,
        scoring="roc_auc",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    xgb_search.fit(X_train, y_train)

xgb_best = xgb_search.best_estimator_
xgb_proba = xgb_best.predict_proba(X_holdout)[:, 1]
xgb_auc = roc_auc_score(y_holdout, xgb_proba)
print("최적 파라미터:", xgb_search.best_params_)
print("홀드아웃 AUC:", round(xgb_auc, 4))


scale_pos_weight 후보 비율: 11.39


/Users/gene/creditLens/ml/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[XGBoost 탐색] 130.1초
최적 파라미터: {'colsample_bytree': np.float64(0.7301321323053057), 'learning_rate': np.float64(0.12271641400994977), 'max_depth': 3, 'min_child_weight': 5, 'n_estimators': 379, 'scale_pos_weight': 5, 'subsample': np.float64(0.9861021229056552)}
홀드아웃 AUC: 0.7613


## 7. AUC 비교 — GMC 대비 격차가 벌어졌는가

In [9]:
GMC_LOGREG_AUC = 0.8598
GMC_XGB_AUC = 0.8691
GMC_GAP = GMC_XGB_AUC - GMC_LOGREG_AUC

hc_gap = xgb_auc - logreg_auc

comparison = pd.DataFrame(
    [
        ("GMC (변수 10개, 전부 수치형)", GMC_LOGREG_AUC, GMC_XGB_AUC, GMC_GAP),
        ("Home Credit 1단계 (변수 121개, 범주형 포함, 단일 테이블)", logreg_auc, xgb_auc, hc_gap),
    ],
    columns=["데이터셋", "로지스틱 회귀 AUC", "XGBoost AUC", "격차(XGB-LR)"],
)
comparison


,데이터셋,로지스틱 회귀 AUC,XGBoost AUC,격차(XGB-LR)
0,"GMC (변수 10개, 전부 수치형)",0.859800,0.869100,0.009300
1,"Home Credit 1단계 (변수 121개, 범주형 포함, 단일 테이블)",0.748669,0.761305,0.012636


In [10]:
print(f"GMC 격차: {GMC_GAP:+.4f}")
print(f"Home Credit 1단계 격차: {hc_gap:+.4f}")
print(f"격차 변화: {hc_gap - GMC_GAP:+.4f} ({'벌어짐' if hc_gap > GMC_GAP else '좁혀짐 또는 유지'})")


GMC 격차: +0.0093
Home Credit 1단계 격차: +0.0126
격차 변화: +0.0033 (벌어짐)


## 8. 고위험 상위 5% 포착 성능 비교

In [11]:
def top_k_metrics(y_true, proba, k_ratio=0.05):
    n = len(y_true)
    k = int(np.ceil(n * k_ratio))
    order = np.argsort(-proba)
    top_idx = order[:k]
    y_true_arr = np.asarray(y_true)
    n_bad_in_top = y_true_arr[top_idx].sum()
    precision = n_bad_in_top / k
    recall = n_bad_in_top / y_true_arr.sum()
    lift = precision / y_true_arr.mean()
    return precision, recall, lift

logreg_p, logreg_r, logreg_l = top_k_metrics(y_holdout, logreg_proba)
xgb_p, xgb_r, xgb_l = top_k_metrics(y_holdout, xgb_proba)

GMC_LOGREG_P, GMC_LOGREG_R = 0.4747, 0.3551
GMC_XGB_P, GMC_XGB_R = 0.4800, 0.3591

top5_table = pd.DataFrame(
    [
        ("GMC", "로지스틱 회귀", GMC_LOGREG_P, GMC_LOGREG_R),
        ("GMC", "XGBoost", GMC_XGB_P, GMC_XGB_R),
        ("Home Credit 1단계", "로지스틱 회귀", logreg_p, logreg_r),
        ("Home Credit 1단계", "XGBoost", xgb_p, xgb_r),
    ],
    columns=["데이터셋", "모델", "정밀도(상위 5%)", "포착률(상위 5%)"],
)
top5_table


,데이터셋,모델,정밀도(상위 5%),포착률(상위 5%)
0,GMC,로지스틱 회귀,0.474700,0.355100
1,GMC,XGBoost,0.480000,0.359100
2,Home Credit 1단계,로지스틱 회귀,0.313069,0.193958
3,Home Credit 1단계,XGBoost,0.343953,0.213092


## 9. XGBoost 피처 중요도 상위 15개

In [12]:
importance = pd.Series(xgb_best.feature_importances_, index=feature_cols).sort_values(ascending=False)
importance.head(15)


EXT_SOURCE_2                                         0.078663
EXT_SOURCE_3                                         0.070207
NAME_INCOME_TYPE_Pensioner                           0.056425
NAME_EDUCATION_TYPE_Higher education                 0.047819
FLAG_DOCUMENT_3                                      0.025394
CODE_GENDER_M                                        0.024477
NAME_INCOME_TYPE_Working                             0.021203
FLOORSMAX_MEDI                                       0.018299
EXT_SOURCE_1                                         0.017075
DAYS_EMPLOYED                                        0.016481
NAME_EDUCATION_TYPE_Secondary / secondary special    0.016126
OWN_CAR_AGE                                          0.015381
WALLSMATERIAL_MODE_Panel                             0.013356
AMT_GOODS_PRICE                                      0.011101
NAME_INCOME_TYPE_State servant                       0.011053
dtype: float32

In [13]:
timing_df = pd.DataFrame([(k, f"{v:.1f}초") for k, v in timings.items()], columns=["단계", "소요 시간"])
print(f"전체 합계: {sum(timings.values()):.1f}초")
timing_df


전체 합계: 189.6초


,단계,소요 시간
0,데이터 로드,0.8초
1,전처리,1.3초
2,로지스틱 회귀 탐색,57.3초
3,XGBoost 탐색,130.1초


## 10. 결론

- 7절 "격차 변화"가 GMC보다 뚜렷하게 커졌다면 → 연구 질문 첫 번째 증거: 복잡도가 늘수록 비선형 모델 우위가 커진다.
- 격차가 GMC와 비슷하거나 더 작다면 → 단일 테이블만으로는 복잡도 증가가 충분하지 않을 수 있음(2단계에서
  다중 테이블 조인 피처까지 추가해 재확인 필요).
- 9절 피처 중요도에서 `EXT_SOURCE_*`가 상위권이면, 이 변수들이 정의가 공개되지 않은 블랙박스성 신용 점수라는 점이
  Phase 7 "SHAP이 스코어카드 수준 설명력을 확보하는가" 검증에서 핵심 쟁점이 될 것이다.
- 결과는 `docs/spike-home-credit.md`에 정리한다.
